# Dixon & Robinson (1998) モデルの実装と StatsBomb データへの適用

Dixon, M. J. and Robinson, M. E. (1998), *A birth process model for association football matches*,
The Statistician, 47(3), 523–538. で提案された、サッカーの得点過程を **2次元の birth process（誕生過程）**
としてモデル化する手法を実装する。

論文で最も当てはまりが良いとされる **モデルVI**（

- チーム別の攻撃力 α・守備力 β、ホームアドバンテージ γ_h
- 前半・後半アディショナルタイムの補正 ρ1, ρ2
- 現在のスコア差に応じた得点率の変化（λ_xy, μ_xy）
- 試合経過とともに得点率が線形に増加する効果 ξ1, ξ2

）を、**退場者（レッドカード）による効果を加えない「純粋な」形**で実装する
（退場者情報自体は使わず、得点イベントのみからモデルを推定する）。

- 入力：`Goals_and_red_cards.csv` を `pandas` で読み込んだ DataFrame（1リーグ・1シーズン分）
- 出力：最尤推定によるチーム別・共通の各パラメータ、対数尤度、AIC、BIC


## 想定するデータ形式

`Goals_and_red_cards.csv` を `pd.read_csv` で読み込んだ df が、以下の列を持つことを想定している
（StatsBomb データから作成された `statsbomb_data/*/*_goals_and_red_cards.csv` の列構成に合わせている）。

| 列名 | 内容 |
|---|---|
| `match_id` | 試合ID |
| `home_team` / `away_team` | ホーム / アウェイチーム名 |
| `home_away` | 得点したチーム（0=home, 1=away）。退場者の行では欠損 |
| `red_card` | 退場者が出たチーム（0=home, 1=away）。得点の行では欠損 |
| `dr_time` | 試合開始からの経過時間を [0, 1] に正規化した値（前半ロスタイムは0.5、後半ロスタイムは1.0に丸め済み） |

本ノートブックは **純粋な Dixon & Robinson モデルVI** を実装するため、`red_card` 列自体は
モデルに使用せず、得点イベント（`home_away` が埋まっている行）のみを使って尤度を構成する。

**重要**: 0-0 の試合については得点イベントの行が存在せず、
`match_id, home_team, away_team` など試合情報のみが埋まった1行（`dr_time` などは欠損値）
として含まれている。下の `prepare_matches` はこの行を「イベント無しの試合」として正しく扱う
（この処理がないと 0-0 の試合が尤度から欠落し、パラメータ推定が歪んでしまう）。


In [10]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize


## モデルの定式化

試合 $k$ における home / away の得点過程を、強度 $\lambda_k(t), \mu_k(t)$ を持つ独立な
非同次ポアソン過程（birth process）とみなす（$t\in[0,1]$ は試合内の正規化時刻）。

基本形（論文 式3.3）:

$$\lambda_k(t) = \alpha_{i(k)}\,\beta_{j(k)}\,\gamma_h, \qquad \mu_k(t) = \alpha_{j(k)}\,\beta_{i(k)}$$

これに

1. スコア状態による倍率 $\lambda_{xy}, \mu_{xy}$（モデルVI: Table 3 相当の4区分＋基準状態）
2. アディショナルタイム補正 $\rho_1, \rho_2$（式3.4：前半・後半終盤の得点強度の上乗せ）
3. 試合経過に伴う線形トレンド $\xi_1, \xi_2$（モデルV/VIの時間変化項）

を組み合わせ、

$$\lambda_k^*(t) = \rho(t)\Big(\alpha_i\beta_j\gamma_h \cdot \lambda_{\text{state}}(\text{score}) + \xi_1 t\Big)$$
$$\mu_k^*(t) = \rho(t)\Big(\alpha_j\beta_i \cdot \mu_{\text{state}}(\text{score}) + \xi_2 t\Big)$$

とする。尤度は式3.5と同じ形（2次元 birth process の尤度）：

$$L_k = \exp(-\Lambda[0,1])\exp(-\Upsilon[0,1]) \prod_l \lambda_k^*(t_l)^{1-J_l}\,\mu_k^*(t_l)^{J_l}$$

対数尤度は「観測された得点イベントでの log 強度の和」から「区間ごとの積分強度 $\Lambda, \Upsilon$ の和」を
引いたもの。退場者による効果は含めない（退場者イベントそのものは無視し、得点イベントのみを扱う）。

識別性の制約は論文と同じく $n^{-1}\sum_i \alpha_i = 1$。実装では
$\alpha_i = \exp(a_i)/\text{mean}(\exp(a))$ という変換で自動的にこの制約を満たすようにしている。


In [11]:
# インジュリータイム（アディショナルタイム）とみなす区間の境界。
# dr_time は 90分を1に正規化した試合内時刻で、前半のロスタイムは0.5に、
# 後半のロスタイムは1.0に丸められている（Dixon & Robinson (1998) と同じ扱い）。
INJ1_START, INJ1_END = 44 / 90, 45 / 90
INJ2_START, INJ2_END = 89 / 90, 90 / 90


class _InvalidRate(Exception):
    """得点強度が非正になってしまった場合に投げる内部例外（最適化の枝刈り用）"""
    pass


def _score_state(x, y):
    """
    スコア状態をモデルVIの4区分＋基準状態(level)に分類する。
      level   : x == y （0-0, 1-1, 2-2, ...）
      home1   : ちょうど 1-0
      away1   : ちょうど 0-1
      homeBig : home が (1-0 を除いて) リード
      awayBig : away が (0-1 を除いて) リード
    """
    if x == y:
        return "level"
    if (x, y) == (1, 0):
        return "home1"
    if (x, y) == (0, 1):
        return "away1"
    if x > y:
        return "homeBig"
    return "awayBig"


## データの前処理

Goals_and_red_cards の df を「試合ごとのイベント列」に変換する。イベントは
`H_GOAL`（home得点）, `A_GOAL`（away得点）の2種類のみを扱い、退場者（`red_card`列）
の行はモデルに使わないため読み飛ばす。


In [12]:
def prepare_matches(df):
    """
    Goals_and_red_cards の df を、試合ごとのイベント列に変換する。
    退場者イベントは純粋なモデルVIでは使わないため無視する。

    戻り値: list of dict、各要素が1試合分
        {
          "home": home_team, "away": away_team,
          "events": [(t, kind), ...]   # kind in {"H_GOAL","A_GOAL"}
        }
    """
    matches = {}
    d = df.sort_values(["match_id", "dr_time"])
    for row in d.itertuples(index=False):
        mid = row.match_id
        if mid not in matches:
            matches[mid] = {"home": row.home_team, "away": row.away_team, "events": []}
        if pd.isna(row.dr_time):
            # 0-0 の試合のプレースホルダー行（イベント無し）
            continue
        if pd.notna(row.red_card):
            # 退場者イベントは純粋なモデルVIでは使わないので読み飛ばす
            continue
        t = float(row.dr_time)
        side = int(row.home_away)  # 0=home, 1=away（得点チーム）
        kind = "H_GOAL" if side == 0 else "A_GOAL"
        matches[mid]["events"].append((t, kind))
    return list(matches.values())


def build_index(matches):
    teams = sorted(set([m["home"] for m in matches] + [m["away"] for m in matches]))
    idx = {t: i for i, t in enumerate(teams)}
    return teams, idx


## 尤度関数

パラメータベクトル `theta` を、チーム別パラメータ（$a_i, b_i$）と共通パラメータ（$\gamma_h, \rho_1,
\rho_2$, スコア状態別倍率, $\xi_1, \xi_2$）に分解し、実際の強度に変換する。

正値制約が必要なパラメータ（$\alpha,\beta,\gamma_h,\rho,\lambda_{xy},\mu_{xy}$）は
すべて `exp()` を通して正値化している。$\xi_1,\xi_2$ は加法項なので実数のまま最適化する。


In [13]:
PARAM_NAMES_GLOBAL = [
    "gamma_h", "rho1", "rho2",
    "lambda_home1", "lambda_away1", "lambda_homeBig", "lambda_awayBig",
    "mu_home1", "mu_away1", "mu_homeBig", "mu_awayBig",
    "xi1", "xi2",
]
N_GLOBAL = len(PARAM_NAMES_GLOBAL)


def unpack_params(theta, n_teams):
    a = theta[0:n_teams]
    b = theta[n_teams:2 * n_teams]
    g = theta[2 * n_teams:]

    alpha = np.exp(a)
    alpha = alpha / alpha.mean()      # 制約 mean(alpha) = 1
    beta = np.exp(b)

    gamma_h = np.exp(g[0])
    rho1 = np.exp(g[1])
    rho2 = np.exp(g[2])
    lam = {"home1": np.exp(g[3]), "away1": np.exp(g[4]),
           "homeBig": np.exp(g[5]), "awayBig": np.exp(g[6]), "level": 1.0}
    mu = {"home1": np.exp(g[7]), "away1": np.exp(g[8]),
          "homeBig": np.exp(g[9]), "awayBig": np.exp(g[10]), "level": 1.0}
    xi1, xi2 = g[11], g[12]
    return alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2


In [14]:
def _check_pos(base, xi, t):
    if base + xi * t <= 0:
        raise _InvalidRate()


def _integrate_rate(t1, t2, base, xi, rho1, rho2):
    """区間 [t1, t2]（スコア状態は一定）で ∫ rho(t)*(base + xi*t) dt を計算する。
    区間内にアディショナルタイムの境界があれば、その前後でrhoを切り替えて積分する。"""
    bpoints = sorted({t1, t2} | {p for p in (INJ1_START, INJ1_END, INJ2_START, INJ2_END) if t1 < p < t2})
    for p in bpoints:
        _check_pos(base, xi, p)
    total = 0.0
    for a, c in zip(bpoints[:-1], bpoints[1:]):
        mid = 0.5 * (a + c)
        if INJ1_START < mid <= INJ1_END:
            r = rho1
        elif INJ2_START < mid <= INJ2_END:
            r = rho2
        else:
            r = 1.0
        total += r * (base * (c - a) + xi * (c ** 2 - a ** 2) / 2.0)
    return total


def _rate_at(t, base, xi, rho1, rho2):
    """イベント発生時刻 t における瞬間強度 rho(t)*(base + xi*t)"""
    val = base + xi * t
    if val <= 0:
        raise _InvalidRate()
    if INJ1_START < t <= INJ1_END:
        r = rho1
    elif INJ2_START < t <= INJ2_END:
        r = rho2
    else:
        r = 1.0
    return r * val


def _match_loglik(match, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2):
    hi, aj = idx[match["home"]], idx[match["away"]]
    lam_k = alpha[hi] * beta[aj] * gamma_h   # home側の基礎強度（スコア=level時）
    mu_k = alpha[aj] * beta[hi]              # away側の基礎強度

    x = y = 0     # スコアの状態
    t_prev = 0.0
    ll = 0.0

    for t_ev, kind in sorted(match["events"], key=lambda e: e[0]):
        s_state = _score_state(x, y)

        base_h = lam_k * lam[s_state]
        base_a = mu_k * mu[s_state]

        # 前のイベントから今回のイベントまでの区間（両者ともゴールが起きなかった＝打ち切り）
        ll -= _integrate_rate(t_prev, t_ev, base_h, xi1, rho1, rho2)
        ll -= _integrate_rate(t_prev, t_ev, base_a, xi2, rho1, rho2)

        if kind == "H_GOAL":
            ll += np.log(_rate_at(t_ev, base_h, xi1, rho1, rho2))
            x += 1
        elif kind == "A_GOAL":
            ll += np.log(_rate_at(t_ev, base_a, xi2, rho1, rho2))
            y += 1
        t_prev = t_ev

    # 最後のイベントから試合終了(t=1)までの打ち切り区間
    s_state = _score_state(x, y)
    base_h = lam_k * lam[s_state]
    base_a = mu_k * mu[s_state]
    ll -= _integrate_rate(t_prev, 1.0, base_h, xi1, rho1, rho2)
    ll -= _integrate_rate(t_prev, 1.0, base_a, xi2, rho1, rho2)
    return ll


def negative_log_likelihood(theta, matches, idx):
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)
    total = 0.0
    for m in matches:
        try:
            total += _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2)
        except _InvalidRate:
            total += -1e9   # 強度が負になるパラメータ領域には大きなペナルティ
    if not np.isfinite(total):
        return 1e12
    return -total


def _count_invalid_matches(theta, matches, idx):
    """
    最適化後のtheta（収束後のパラメータ）で、実際に強度が非正になってしまう
    試合が何件あるかを数える。0件でなければ、対数尤度・AIC/BICは
    -1e9ペナルティが混入した信頼できない値になっている。
    """
    n_teams = len(idx)
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)
    n_invalid = 0
    for m in matches:
        try:
            _match_loglik(m, idx, alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2)
        except _InvalidRate:
            n_invalid += 1
    return n_invalid


## 初期値の設定

チームごとの平均得点・失点からラフな攻撃力・守備力の初期値を作ることで、最適化の収束を安定させる。


In [15]:
def initial_theta(matches, idx):
    n_teams = len(idx)
    goals_for = np.zeros(n_teams)
    goals_against = np.zeros(n_teams)
    games = np.zeros(n_teams)
    total_goals = 0
    total_games = 0
    home_goals_total = 0
    away_goals_total = 0
    for m in matches:
        hi, aj = idx[m["home"]], idx[m["away"]]
        xg = sum(1 for t, k in m["events"] if k == "H_GOAL")
        yg = sum(1 for t, k in m["events"] if k == "A_GOAL")
        goals_for[hi] += xg
        goals_against[aj] += xg
        goals_for[aj] += yg
        goals_against[hi] += yg
        games[hi] += 1
        games[aj] += 1
        total_goals += xg + yg
        total_games += 1
        home_goals_total += xg
        away_goals_total += yg

    avg = total_goals / max(2 * total_games, 1)
    with np.errstate(divide="ignore", invalid="ignore"):
        att0 = np.where(games > 0, (goals_for / np.maximum(games, 1)) / avg, 1.0)
        def0 = np.where(games > 0, (goals_against / np.maximum(games, 1)) / avg, 1.0)
    att0 = np.clip(att0, 0.3, 3.0)
    def0 = np.clip(def0, 0.3, 3.0)

    a0 = np.log(att0)
    b0 = np.log(def0)

    gamma0 = home_goals_total / max(away_goals_total, 1)
    g0 = np.zeros(N_GLOBAL)
    g0[0] = np.log(max(gamma0, 0.1))   # gamma_h の初期値、他は倍率1(log=0)からスタート
    return np.concatenate([a0, b0, g0])


## 推定（最適化）

パラメータ数が多く（チーム数20なら 2×20+13=53 個）尤度曲面が悪条件になりやすいため、

1. `scipy.optimize.minimize(method="L-BFGS-B")` でおおまかに尤度を改善
2. その結果を初期値として `method="Powell"`（微分不要）で仕上げる

という2段階の最適化を行う。L-BFGS-B単体では `ABNORMAL_TERMINATION_IN_LNSRCH` で
途中停止することが多いが、その後Powell法で仕上げることで安定して収束することを
実データ（Premier League 2015-16など）で確認している。


In [19]:
def fit_dixon_robinson(df, maxiter=1000, disp=False):
    """
    Goals_and_red_cards の df に 純粋な Dixon & Robinson (1998) モデルVI
    （退場者効果なし）をあてはめ、各パラメータ・対数尤度・AIC・BICを返す。

    Parameters
    ----------
    df : pandas.DataFrame
        1リーグ・1シーズン分の Goals_and_red_cards.csv を読み込んだもの。
    maxiter : int
        最適化の反復回数の目安。
    disp : bool
        Trueなら各最適化ステージの途中経過を表示する。

    Returns
    -------
    summary : dict
        n_matches, n_teams, n_params, log_likelihood, AIC, BIC, converged, message。
        収束後のパラメータでも一部の試合で得点強度が非正になり-1e9ペナルティが
        混入している場合は、messageの先頭に[警告]として明記される
        （この場合、log_likelihood/AIC/BICの値は信頼できない）。
    team_params : pandas.DataFrame
        チーム別の攻撃力(alpha_attack)・守備力(beta_defence)
    global_params : pandas.DataFrame
        共通パラメータ（ホームアドバンテージ、アディショナルタイム補正、
        スコア状態別倍率、時間トレンド）
    res : scipy.optimize.OptimizeResult
        最終ステージ(Powell)の最適化結果
    """
    matches = prepare_matches(df)
    teams, idx = build_index(matches)
    n_teams = len(teams)

    theta0 = initial_theta(matches, idx)

    # 発散を防ぐための緩いbounds（team別 alpha, beta と共通パラメータ、いずれもlogスケール）
    bounds = [(-3, 3)] * n_teams + [(-3, 3)] * n_teams
    bounds += [(-3, 3)]            # gamma_h
    bounds += [(-3, 3), (-3, 3)]   # rho1, rho2
    bounds += [(-3, 3)] * 4        # lambda_home1, lambda_away1, lambda_homeBig, lambda_awayBig
    bounds += [(-3, 3)] * 4        # mu_home1, mu_away1, mu_homeBig, mu_awayBig
    bounds += [(-5, 5), (-5, 5)]   # xi1, xi2 (加法項なのでlogスケールではない)

    res1 = minimize(
        negative_log_likelihood, theta0, args=(matches, idx),
        method="L-BFGS-B", bounds=bounds,
        options={"maxiter": maxiter, "maxfun": maxiter * 50},
    )
    res = minimize(
        negative_log_likelihood, res1.x, args=(matches, idx),
        method="Powell", bounds=bounds,
        options={"maxiter": maxiter * 20, "maxfev": maxiter * 200, "xtol": 1e-10, "ftol": 1e-12},
    )
    if disp:
        print(f"[stage1: L-BFGS-B] loglik={-res1.fun:.4f} success={res1.success}")
        print(f"[stage2: Powell]   loglik={-res.fun:.4f} success={res.success}")

    theta = res.x
    alpha, beta, gamma_h, rho1, rho2, lam, mu, xi1, xi2 = unpack_params(theta, n_teams)

    n_params = len(theta)
    loglik = -res.fun
    aic = 2 * n_params - 2 * loglik
    # BIC の n は「試合数」を単位として計算している（尤度が試合ごとの積であるため）。
    # イベント数を単位にしたい場合は下の len(matches) を合計イベント数に置き換えること。
    bic = n_params * np.log(len(matches)) - 2 * loglik

    team_params = pd.DataFrame({"team": teams, "alpha_attack": alpha, "beta_defence": beta})
    global_params = pd.DataFrame({
        "parameter": [
            "gamma_h", "rho1", "rho2",
            "lambda_home1", "lambda_away1", "lambda_homeBig", "lambda_awayBig",
            "mu_home1", "mu_away1", "mu_homeBig", "mu_awayBig",
            "xi1", "xi2",
        ],
        "estimate": [
            gamma_h, rho1, rho2,
            lam["home1"], lam["away1"], lam["homeBig"], lam["awayBig"],
            mu["home1"], mu["away1"], mu["homeBig"], mu["awayBig"],
            xi1, xi2,
        ],
    })

    n_invalid = _count_invalid_matches(theta, matches, idx)
    message = str(res.message)
    if n_invalid > 0:
        message = (
        f"[WARNING] Even after convergence, {n_invalid} match(es) still have "
        f"non-positive scoring intensity, so a -1e9 penalty has leaked into "
        f"the log-likelihood. The log-likelihood/AIC/BIC values are not "
        f"reliable. " + message
    )

    summary = {
        "n_matches": len(matches), "n_teams": n_teams, "n_params": n_params,
        "log_likelihood": loglik, "AIC": aic, "BIC": bic,
        "converged": bool(res.success), "message": message,
    }

    print("==== モデル適合結果 ====")
    print(f"試合数: {summary['n_matches']}, チーム数: {summary['n_teams']}, パラメータ数: {summary['n_params']}")
    print(f"対数尤度: {summary['log_likelihood']:.3f}")
    print(f"AIC: {summary['AIC']:.3f}")
    print(f"BIC: {summary['BIC']:.3f}")
    print(f"収束: {summary['converged']} ({summary['message']})")
    print()
    print("---- チーム別パラメータ ----")
    print(team_params.to_string(index=False))
    print()
    print("---- 共通パラメータ ----")
    print(global_params.to_string(index=False))

    return summary, team_params, global_params, res


## 使い方（1ファイルを試す）

このノートブックが `~/修士課程/BP/Models/` に置かれている想定で、
`../statsbomb_data/` 以下のcsvを相対パスで読み込む。


In [20]:
df = pd.read_csv("../statsbomb_data/Premier_League/PL2015-2016_goals_and_red_cards.csv")
summary, team_params, global_params, res = fit_dixon_robinson(df, disp=True)


[stage1: L-BFGS-B] loglik=-555.5935 success=True
[stage2: Powell]   loglik=-555.5935 success=True
==== モデル適合結果 ====
試合数: 380, チーム数: 20, パラメータ数: 53
対数尤度: -555.594
AIC: 1217.187
BIC: 1426.016
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                team  alpha_attack  beta_defence
     AFC Bournemouth      0.853595      1.108864
             Arsenal      1.340754      0.514741
         Aston Villa      0.347676      1.239235
             Chelsea      1.218281      0.774309
      Crystal Palace      0.670022      0.762341
             Everton      1.231369      0.857065
      Leicester City      1.398502      0.450133
           Liverpool      1.299154      0.726263
     Manchester City      1.516648      0.652527
   Manchester United      0.936050      0.482139
    Newcastle United      0.792760      1.050016
        Norwich City      0.711203      1.058151
         Southampton      1.170900      0.557423
          Stoke City      0.781877      0.817060
     

## 全リーグ・全シーズンへの一括適用

`statsbomb_data` 以下の `*_goals_and_red_cards.csv` を自動的に探して、それぞれにモデルを
あてはめ、AIC・BICなどをまとめた一覧表を作る（各ファイル数十秒〜数分かかることがある）。
チーム別・共通パラメータもリーグ・シーズンごとにcsvへ保存する。


In [21]:
import glob
import os

DATA_DIR = "../statsbomb_data"
OUT_DIR = "./dixon_robinson_pure_results"
os.makedirs(OUT_DIR, exist_ok=True)

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_goals_and_red_cards.csv"), recursive=True))
print(f"{len(csv_paths)} 件のcsvが見つかりました")

results_all = []
for path in csv_paths:
    print("=" * 60)
    print(path)
    df_i = pd.read_csv(path)
    league = df_i["competition_name"].iloc[0] if "competition_name" in df_i.columns and len(df_i) else os.path.basename(path)
    season = df_i["season_name"].iloc[0] if "season_name" in df_i.columns and len(df_i) else ""
    tag = f"{league}_{season}".replace("/", "-").replace(" ", "_")
    try:
        summary_i, team_params_i, global_params_i, _ = fit_dixon_robinson(df_i, disp=True)
        summary_i["league"] = league
        summary_i["season"] = season
        summary_i["file"] = os.path.basename(path)
        results_all.append(summary_i)
        team_params_i.to_csv(os.path.join(OUT_DIR, f"team_params_{tag}.csv"), index=False)
        global_params_i.to_csv(os.path.join(OUT_DIR, f"global_params_{tag}.csv"), index=False)
    except Exception as e:
        print("失敗:", e)

summary_all_df = pd.DataFrame(results_all)
summary_all_df.to_csv(os.path.join(OUT_DIR, "summary_all.csv"), index=False)
summary_all_df


11 件のcsvが見つかりました
../statsbomb_data/FA_Women's_Super_League/WSL2018-2019_goals_and_red_cards.csv
[stage1: L-BFGS-B] loglik=-99.5337 success=True
[stage2: Powell]   loglik=-99.5337 success=True
==== モデル適合結果 ====
試合数: 107, チーム数: 11, パラメータ数: 35
対数尤度: -99.534
AIC: 269.067
BIC: 362.616
収束: True (Optimization terminated successfully.)

---- チーム別パラメータ ----
                      team  alpha_attack  beta_defence
               Arsenal WFC      2.282710      0.686594
       Birmingham City WFC      0.954218      0.830367
Brighton & Hove Albion WFC      0.558189      1.732132
          Bristol City WFC      0.572884      1.622833
               Chelsea FCW      1.321441      0.633739
               Everton LFC      0.524138      1.591930
             Liverpool WFC      0.709887      1.731127
       Manchester City WFC      1.726996      0.856166
               Reading WFC      1.161912      1.504190
       West Ham United LFC      0.802755      1.646773
           Yeovil Town LFC      0.384871    

,n_matches,n_teams,n_params,log_likelihood,AIC,BIC,converged,message,league,season,file
0,107,11,35,-9.953372e+01,2.690674e+02,3.626164e+02,True,Optimization terminated successfully.,FA Women's Super League,2018/2019,WSL2018-2019_goals_and_red_cards.csv
1,87,12,37,-6.041685e+01,1.948337e+02,2.860723e+02,True,Optimization terminated successfully.,FA Women's Super League,2019/2020,WSL2019-2020_goals_and_red_cards.csv
2,131,12,37,-9.335717e+01,2.607143e+02,3.670966e+02,True,Optimization terminated successfully.,FA Women's Super League,2020/2021,WSL2020-2021_goals_and_red_cards.csv
3,132,12,37,-7.978453e+01,2.335691e+02,3.402327e+02,True,Optimization terminated successfully.,FA Women's Super League,2023/2024,WSL2023-2024_goals_and_red_cards.csv
4,132,12,37,-1.000000e+09,2.000000e+09,2.000000e+09,True,"[WARNING] Even after convergence, 1 match(es) ...",Frauen Bundesliga,2023/2024,FB2023-2024_goals_and_red_cards.csv
5,115,11,35,-1.218186e+02,3.136372e+02,4.097098e+02,True,Optimization terminated successfully.,Indian Super league,2021/2022,ISL2021-2022_goals_and_red_cards.csv
6,380,20,53,-5.222177e+02,1.150435e+03,1.359264e+03,True,Optimization terminated successfully.,La Liga,2015/2016,LL2015-2016_goals_and_red_cards.csv
7,240,16,45,-1.622868e+02,4.145735e+02,5.712023e+02,True,Optimization terminated successfully.,Liga F,2023/2024,LF2023-2024_goals_and_red_cards.csv
8,377,20,53,-5.810269e+02,1.268054e+03,1.476463e+03,True,Optimization terminated successfully.,Ligue 1,2015/2016,L12015-2016_goals_and_red_cards.csv
9,380,20,53,-5.555935e+02,1.217187e+03,1.426016e+03,True,Optimization terminated successfully.,Premier League,2015/2016,PL2015-2016_goals_and_red_cards.csv


## 注意点・拡張のヒント

- **0-0の試合**: `Goals_and_red_cards.csv` にプレースホルダー行として含まれている前提で
  実装している。もし別データソースで「イベントの行が丸ごと無い」形式になっている場合は、
  別途「試合一覧（home_team, away_team）」を用意して `prepare_matches` に事前にマージ
  しておくこと。
- **退場者（レッドカード）の効果**は本ノートブックでは意図的にモデル化していない
  （`red_card` 列の行は無視し、得点イベントのみで尤度を構成する）。退場者による
  数的優位・不利の影響を加えたい場合は、別途 δ_adv, δ_disadv のようなスコア状態と
  同様の倍率パラメータを追加する拡張が考えられる。
- **サンプルサイズが小さいリーグ・シーズン**（1シーズン・1リーグのみ）では、
  特に `rho1, rho2`（アディショナルタイム補正）のようなレアなイベントに依存する
  パラメータは推定が不安定になりやすい。複数シーズンをプールする、あるいは
  これらのパラメータを固定 (=1) して尤度比検定で必要性を確認する、といった対応が
  考えられる。
- **標準誤差**は本実装では出力していない（AIC/BICとパラメータ点推定のみ）。
  必要であれば最適化後の数値ヘシアン（例: `scipy.optimize`の`hess_inv`や
  `numdifftools`）から近似的に計算できる。
